### Figure 3

In [ ]:
%cd ../..
import os
import glob
import nibabel as nib
import numpy as np
import nilearn
import nilearn.plotting as plotting
import trimesh
import tempfile
import subprocess
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

plt.rcParams['savefig.pad_inches'] = 0

In [ ]:
tmpdir = tempfile.mkdtemp()
threads = 16


# patient positive at all 3 conditions
# visually 3T MRI positive, histophatological FCD 2b
#site, subj_id, selected_display_mode = '7T_rory', 'sub-55', 'z'
#crop_margin = 80

# patient only positive at 7T
# visually 3T MRI negative, radiological FCD at 7T
#site, subj_id, selected_display_mode = 'cambridge', 'sub-010', 'y'
#crop_margin = 60

# patient negative at all 3 conditions
# visually 3T MRI negative, 7T MRI positive, histophatological FCD
#site, subj_id, selected_display_mode = 'RICE', 'sub-RICE021', 'z'
#crop_margin = 60

# control with false positives at 7T
# no lesion mask, so the slices are centred on the largest predicted cluster of each condition
#site, subj_id, selected_display_mode = 'bonn_7T', 'sub-00115', 'y'
#crop_margin = 80

# control with false positives at 7T
# no lesion mask, so the slices are centred on the largest predicted cluster of each condition
#site, subj_id, selected_display_mode = 'bonn_7T', 'sub-00110', 'z'
#crop_margin = 80

# control with false positives at 7T
# no lesion mask, so the slices are centred on the largest predicted cluster of each condition
site, subj_id, selected_display_mode = 'RICE', 'sub-RICE019', 'y'
crop_margin = 80

In [ ]:
# conditions this subject has a MELD-graph prediction for
conditions = sorted(path.split('/')[3] for path in
                    glob.glob(f'data/meld_graph/{site}/*/output/predictions_reports/{subj_id}/predictions/prediction.nii.gz'))


b0_uhfadp = next(b0 for b0 in ['7T', '7T_CP', '7T_pTx'] if b0 in conditions)

meldgraph_dir_uhfadp = f'data/meld_graph/{site}/{b0_uhfadp}/'
meldgraph_dir_uhfdef = f'data/meld_graph/{site}/{b0_uhfadp}_nouhf/'
meldgraph_dir_3T = f'data/meld_graph/{site}/3T/'

# not every site has a FLAIR at both field strengths (cambridge has no 3T FLAIR), and not
# every subject of a site with one has been run in that condition
b0s_flair = [b0 for b0 in ['3T_FLAIR', f'{b0_uhfadp}_FLAIR'] if b0 in conditions]
meldgraph_dirs_flair = {b0: f'data/meld_graph/{site}/{b0}/' for b0 in b0s_flair}

# the lesion mask is drawn on the 7T T1w, so where a site has several it has to be the one
# of the acquisition used as the reference
lesion_masks = sorted(glob.glob(f'data/raw/{site}/bids/derivatives/lesion_masks_coreg/{subj_id}_ses-7*_lesionmask.nii.gz'))
if len(lesion_masks) > 1:
    if b0_uhfadp.startswith('7T_'):
        lesion_masks = [path for path in lesion_masks if f"acq-{b0_uhfadp.removeprefix('7T_')}" in path]
    else:
        lesion_masks = [path for path in lesion_masks if "rec-offline" in path]
assert len(lesion_masks) <= 1, f'expected at most one lesion mask for {subj_id}, found {lesion_masks}'
# controls have no lesion mask, the slices are then centred on the predicted clusters instead
lesion_mask_uhfadp_path = lesion_masks[0] if lesion_masks else None

print(f'conditions found: {conditions}')
print(f'comparing {b0_uhfadp} against {["3T", b0_uhfadp + "_nouhf"] + b0s_flair}')
print(f'lesion mask: {lesion_mask_uhfadp_path or "none, treating this subject as a control"}')

In [ ]:
def crop(img, center, margin):
    # Reorient to canonical RAS so center is interpreted consistently
    img = nib.as_closest_canonical(img)
    shape = np.array(img.shape[:3])

    ras2vox = np.linalg.inv(img.affine)
    center_vox = nib.affines.apply_affine(ras2vox, center)

    margin = np.broadcast_to(np.asarray(margin, dtype=float), (3,))

    # Desired window (may fall outside the volume)
    i0 = np.floor(center_vox - margin).astype(int)
    i1 = np.ceil(center_vox + margin).astype(int)

    # Slide the window back inside the volume, preserving its size,
    # so an off-center point crops less from the opposite side.
    under = np.maximum(-i0, 0)
    i0 += under
    i1 += under

    over = np.maximum(i1 - shape, 0)
    i0 -= over
    i1 -= over

    # Only clamp (shrink) if the window is larger than the axis itself
    i0 = np.maximum(i0, 0)
    i1 = np.minimum(i1, shape)

    data = np.asanyarray(img.dataobj)[i0[0]:i1[0], i0[1]:i1[1], i0[2]:i1[2]]

    # Robust affine update: world_new = world_old @ T(offset_in_voxels)
    T = np.eye(4)
    T[:3, 3] = i0
    new_affine = img.affine @ T

    return nib.Nifti1Image(data, new_affine)

In [ ]:
def add_surf_overlay(plotting_display, 
                     fs_surface,
                     surfdata = None, 
                     vmin = None, 
                     vmax = None,
                     colormap = 'viridis',
                     cortexlabel = None,
                     color='C0'):
    coords, faces = fs_surface

    for label, ax in plotting_display.axes.items():
        direction = ax.direction
        coord = ax.coord
        match direction:
            case 'x':
                plane_origin = [coord, 0, 0]
                plane_normal = [1, 0, 0]
                select_slices = [1, 2]
            case 'y':
                plane_origin = [0, coord, 0]
                plane_normal = [0, 1, 0]
                select_slices = [0, 2]
            case 'z':
                plane_origin = [0, 0, coord]
                plane_normal = [0, 0, 1]
                select_slices = [0, 1]

        mesh = trimesh.Trimesh(vertices=coords, faces=faces, process=False)
        section, face_index = trimesh.intersections.mesh_plane(mesh, plane_origin=plane_origin, plane_normal=plane_normal, return_faces=True)
        if surfdata is not None:
            colors = surfdata[faces[face_index]].mean(axis=1)
            if vmin is None:
                vmin = colors.min()
            if vmax is None:
                vmax = colors.max()
            norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
            colors = norm(colors)
            colors = plt.get_cmap(colormap)(colors)
            if cortexlabel is not None:
                alpha = np.all(np.isin(faces[face_index],cortexlabel),axis=1)
                colors[:, -1] = alpha
        else:
            colors=color
        lc = LineCollection(section[:,:,select_slices], colors=colors)
        ax.ax.add_collection(lc)

def tkreg_to_scanner_affine(c_ras):
    # affine for converting from surface RAS (freesurfer tkreg) to scanner RAS
    # the c_ras translation is the only thing that matters
    c_ras = c_ras
    tkreg_to_scanner_affine = np.eye(4)
    tkreg_to_scanner_affine[:3, 3] = c_ras
    return tkreg_to_scanner_affine

def read_lta_ras2ras_affine(path_lta):
    with open(path_lta, 'r') as f:
        lines = f.readlines()

    # determine LTA type (0 for vox2vox, 1 is ras2ras)
    for line in lines:
        if line.startswith('#'):
            continue
        if '=' in line:
            type = int(line.split('=')[1].strip()[0])
            break

    if type not in [0, 1]:
        raise ValueError(f'Unsupported LTA type {type} in {path_lta}')
    
    # The affine matrix is 4 lines after a line that starts with "1 4 4"
    start_index = None
    for i, line in enumerate(lines):
        if line.startswith('1 4 4'):
            start_index = i + 1
            break
    affine_lines = lines[start_index:start_index+4]
    affine = np.loadtxt(affine_lines)

    if type == 1:
        return affine
    
    # for vox2vox, we need to convert from voxel to RAS coordinates
    def parse_volume_info_to_affine(lines, search_string=None):
        if not search_string in ['src volume info', 'dst volume info']:
            raise ValueError(f'search_string must be either "src volume info" or "dst volume info", got {search_string}')
        
        start_index = None
        for i, line in enumerate(lines):
            if line.startswith(search_string):
                start_index = i + 3
                break
        volume_info_lines = lines[start_index:start_index+6]

        volume_data = np.loadtxt([line.split("=")[1] for line in volume_info_lines])
        dims, deltas, dir_cos, center_ras = (
            volume_data[0],
            volume_data[1],
            volume_data[2:5],
            volume_data[5],
        )
        dir_cos_delta = dir_cos.T * deltas
        vol_center = (dir_cos_delta @ dims[:3]) / 2
        affine = np.eye(4)
        affine[:3, :3] = dir_cos_delta
        affine[:3, 3] = center_ras - vol_center
        return affine

    src_affine = parse_volume_info_to_affine(lines, search_string='src volume info')
    dst_affine = parse_volume_info_to_affine(lines, search_string='dst volume info')

    ras2ras = dst_affine @ affine @ np.linalg.inv(src_affine)
    return ras2ras

def add_surf_overlay_fromfile_coreg(plotting_display, path_surface, path_lta=None, ras2ras=None, **kwargs):
    coords, faces, metadata = nib.freesurfer.io.read_geometry(path_surface, read_metadata=True)

    coords = nib.affines.apply_affine(tkreg_to_scanner_affine(metadata['cras']), coords)
    if path_lta is not None:
        lta_affine = read_lta_ras2ras_affine(path_lta)
        coords = nib.affines.apply_affine(lta_affine, coords)
    if ras2ras is not None:
        coords = nib.affines.apply_affine(ras2ras, coords)
    
    add_surf_overlay(plotting_display=plotting_display,
                     fs_surface=(coords, faces),
                     **kwargs
                    )

In [ ]:
# load reference input image, prediction and plot
input_uhfadp_path = glob.glob(f'{meldgraph_dir_uhfadp}/input/{subj_id}/**/*T1w.nii.gz', recursive=True) + glob.glob(f'{meldgraph_dir_uhfadp}/input/{subj_id}/**/*UNIT1.nii.gz', recursive=True)
input_uhfadp_path = input_uhfadp_path[0]
input_uhfadp = nib.load(input_uhfadp_path)
input_uhfadp_data = input_uhfadp.get_fdata()

pred_uhfadp_path = f'{meldgraph_dir_uhfadp}/output/predictions_reports/{subj_id}/predictions/prediction.nii.gz'
pred_uhfadp = nib.load(pred_uhfadp_path)
pred_uhfadp_data = pred_uhfadp.get_fdata()

lesion_mask_uhfadp = nib.load(lesion_mask_uhfadp_path) if lesion_mask_uhfadp_path else None
if lesion_mask_uhfadp is not None:
    # check that lesion mask is in same space as input and prediction
    assert np.allclose(lesion_mask_uhfadp.affine, input_uhfadp.affine, atol=0.1)
    assert np.allclose(lesion_mask_uhfadp.affine, pred_uhfadp.affine, atol=0.1)

def deoblique(img):
    """Replace an image's oblique affine by an axis-aligned one, keeping the FOV centre in place.

    Returns the de-obliqued image and the RAS-to-RAS transform from the original scanner
    space into the de-obliqued one, which anything else defined in the original space
    (i.e. the FreeSurfer surfaces) has to be pushed through as well.
    """
    img = nib.as_closest_canonical(img)  # otherwise the positive diagonal below silently flips axes
    new = np.eye(4)
    new[:3, :3] = np.diag(np.linalg.norm(img.affine[:3, :3], axis=0))
    centre_vox = (np.array(img.shape[:3]) - 1) / 2
    new[:3, 3] = nib.affines.apply_affine(img.affine, centre_vox) - new[:3, :3] @ centre_vox
    out = nib.Nifti1Image(img.dataobj, new, img.header)
    out.set_sform(new, code=1)
    out.set_qform(new, code=1)
    return out, new @ np.linalg.inv(img.affine)

input_uhfadp, deoblique_uhfadp = deoblique(input_uhfadp)
pred_uhfadp, _ = deoblique(pred_uhfadp)
if lesion_mask_uhfadp is not None:
    lesion_mask_uhfadp, _ = deoblique(lesion_mask_uhfadp)

# save to tmpdir so that coregistration uses the de-obliqued images
input_uhfadp_path = os.path.join(tmpdir, 'input_uhfadp.nii.gz')
nib.save(input_uhfadp, input_uhfadp_path)



In [ ]:
def find_input(meldgraph_dir_comp, subj_id, patterns=('*T1w.nii.gz', '*UNIT1.nii.gz')):
    for pattern in patterns:
        hits = sorted(glob.glob(f'{meldgraph_dir_comp}/input/{subj_id}/**/{pattern}', recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f'no input matching {patterns} in {meldgraph_dir_comp}/input/{subj_id}')


def load_coreg_comp(subj_id, meldgraph_dir_comp, input_uhfadp_path, tag, input_patterns=('*T1w.nii.gz', '*UNIT1.nii.gz')):
    # coregister comp input and prediction to ref space
    # input_patterns picks the image to show, the prediction always lives in the space of the
    # condition's T1w input, so for a FLAIR condition the two need a registration each
    #input_comp_path = f'{meldgraph_dir_comp}/output/fs_outputs/{subj_id}/mri/orig.mgz'
    input_comp_path = find_input(meldgraph_dir_comp, subj_id, input_patterns)
    t1w_comp_path = find_input(meldgraph_dir_comp, subj_id)
    pred_comp_path = f'{meldgraph_dir_comp}/output/predictions_reports/{subj_id}/predictions/prediction.nii.gz'

    # own output dir per comparison, otherwise the second call overwrites the first one's files
    outdir = os.path.join(tmpdir, tag)
    os.makedirs(outdir, exist_ok=True)

    print(f'Coregistering {input_comp_path} and\n {pred_comp_path} to\n {input_uhfadp_path}')

    #######
    # skullstrip coregistration input to make this robust for MP2RAGE
    ref_input_stripped_path = os.path.join(tmpdir, 'ref_input_stripped.nii.gz')
    if not os.path.exists(ref_input_stripped_path):  # the reference is the same for every comparison
        cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "mri_synthstrip -i {input_uhfadp_path} -o {ref_input_stripped_path} --no-csf"'
        subprocess.run(cmd, shell=True, check=True)

    comp_input_stripped_path = os.path.join(outdir, 'comp_input_stripped.nii.gz')
    cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "mri_synthstrip -i {input_comp_path} -o {comp_input_stripped_path} --no-csf"'
    subprocess.run(cmd, shell=True, check=True)

    #######
    # using mri_coreg
    cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "FS_LICENSE={os.getcwd()}/license.txt mri_coreg --mov {comp_input_stripped_path} --ref {ref_input_stripped_path} --reg {outdir}/reg.lta --threads {threads} --dof 12"'
    subprocess.run(cmd, shell=True, check=True)

    cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "FS_LICENSE={os.getcwd()}/license.txt mri_vol2vol --mov {input_comp_path} --targ {ref_input_stripped_path} --o {outdir}/input_comp_coreg.nii.gz --lta {outdir}/reg.lta --trilin"'
    subprocess.run(cmd, shell=True, check=True)

    # the prediction sits in the space of the condition's T1w, which for a FLAIR condition is
    # not the image registered above
    pred_lta_path = os.path.join(outdir, 'reg.lta')
    if t1w_comp_path != input_comp_path:
        pred_lta_path = os.path.join(outdir, 'reg_t1w.lta')
        t1w_stripped_path = os.path.join(outdir, 'comp_t1w_stripped.nii.gz')
        cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "mri_synthstrip -i {t1w_comp_path} -o {t1w_stripped_path} --no-csf"'
        subprocess.run(cmd, shell=True, check=True)

        cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "FS_LICENSE={os.getcwd()}/license.txt mri_coreg --mov {t1w_stripped_path} --ref {ref_input_stripped_path} --reg {pred_lta_path} --threads {threads} --dof 12"'
        subprocess.run(cmd, shell=True, check=True)

    cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "FS_LICENSE={os.getcwd()}/license.txt mri_vol2vol --mov {pred_comp_path} --targ {ref_input_stripped_path} --o {outdir}/pred_comp_coreg.nii.gz --lta {pred_lta_path} --nearest"'
    subprocess.run(cmd, shell=True, check=True)

    #######
    # using mri_easyreg
    #threads = 1
    #cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "mri_easyreg --flo {comp_input_stripped_path} --ref {ref_input_stripped_path} --fwd_field {tmpdir}/fwd_field.nii.gz --flo_seg {tmpdir}/flo_seg.nii.gz --ref_seg {tmpdir}/ref_seg.nii.gz --threads {threads} --affine_only"'
    #subprocess.run(cmd, shell=True, check=True)

    ## warp input
    #cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "mri_easywarp --i {input_comp_path[0]} --o {outdir}/input_comp_coreg.nii.gz --field {tmpdir}/fwd_field.nii.gz --threads {threads}"'
    #subprocess.run(cmd, shell=True, check=True)

    ## warp prediction
    #cmd = f'apptainer exec freesurfer_8.1.0.sif /bin/bash -c "mri_easywarp --i {pred_comp_path} --o {outdir}/pred_comp_coreg.nii.gz --field {tmpdir}/fwd_field.nii.gz --threads {threads}"'
    #subprocess.run(cmd, shell=True, check=True)

    print(f'Coregistered volumes saved to {outdir}')

    input_comp_coreg_path = os.path.join(outdir, 'input_comp_coreg.nii.gz')
    pred_comp_coreg_path = os.path.join(outdir, 'pred_comp_coreg.nii.gz')

    # the surfaces are in the T1w space as well, so they go through the prediction's transform
    return input_comp_coreg_path, pred_comp_coreg_path, pred_lta_path


input_3T_coreg_path, pred_3T_coreg_path, affine_3T_path = load_coreg_comp(subj_id, meldgraph_dir_3T, input_uhfadp_path, tag='3T')
input_uhfdef_path, pred_uhfdef_path, affine_uhfdef_path = load_coreg_comp(subj_id, meldgraph_dir_uhfdef, input_uhfadp_path, tag='uhfdef')

coreg_flair_paths = {b0: load_coreg_comp(subj_id, meldgraph_dir, input_uhfadp_path, tag=b0, input_patterns=('*FLAIR.nii.gz',))
                     for b0, meldgraph_dir in meldgraph_dirs_flair.items()}

In [ ]:
input_3T_coreg = nib.load(input_3T_coreg_path)
pred_3T_coreg = nib.load(pred_3T_coreg_path)

input_uhfdef_coreg = nib.load(input_uhfdef_path)
pred_uhfdef_coreg = nib.load(pred_uhfdef_path)

coreg_flair = {b0: (nib.load(input_path), nib.load(pred_path))
               for b0, (input_path, pred_path, _) in coreg_flair_paths.items()}

In [ ]:
# every condition in one list, the UHF adapted reference last so it is plotted after its comparisons
plot_conditions = ([('3T', input_3T_coreg, pred_3T_coreg),
                    (f'{b0_uhfadp}_nouhf', input_uhfdef_coreg, pred_uhfdef_coreg)]
                   + [(b0, input_flair, pred_flair) for b0, (input_flair, pred_flair) in coreg_flair.items()]
                   + [(b0_uhfadp, input_uhfadp, pred_uhfadp)])

# a patient gets one slice through the lesion, a control one slice per condition through the
# centre of that condition's largest predicted cluster (all predictions are in reference space)
if lesion_mask_uhfadp is not None:
    # after de-obliquing the lesion sits at different world coordinates, so cut it now
    centres = {'lesion': plotting.find_xyz_cut_coords(
        nib.Nifti1Image((lesion_mask_uhfadp.get_fdata() > 0).astype(np.float32), lesion_mask_uhfadp.affine))}
else:
    # find_xyz_cut_coords centres on the largest connected component of the binarised prediction
    centres = {f'largest cluster {label}': plotting.find_xyz_cut_coords(
                   nib.Nifti1Image((pred.get_fdata() > 0).astype(np.float32), pred.affine))
               for label, _, pred in plot_conditions if (pred.get_fdata() > 0).any()}
    assert centres, f'{subj_id} has no lesion mask and no prediction in any condition, nothing to centre on'

print(f'centring slices on: {list(centres)}')


In [ ]:

fig_height = 5
low_percentile = 1
high_percentile = 99

other_plot_opts = dict(
    draw_cross=False,
    annotate=False,
    colorbar=False,
    radiological=True,
    resampling_interpolation='nearest',
)

# cut_coords is the loop variable, so the surface overlays of the last cell are drawn at the
# last centre. For a patient that is the lesion, for a control any of the clusters will do.
for centre_label, cut_coords in centres.items():
    print(f'==== centred on {centre_label} ====')
    selected_cut_coords = [cut_coords[{'x': 0, 'y': 1, 'z': 2}[selected_display_mode]]]

    # the slices are drawn with equal aspect, so a figure of a different aspect ratio than the
    # cropped block keeps a border of its background colour at the sides. The crop clips at the
    # edge of the FOV, so the block is not square in general. Every comparison is resampled onto
    # the reference grid, so one size fits all panels of this centre, the last cell included.
    cropped_ref = crop(input_uhfadp, cut_coords, crop_margin)
    cropped_extent = np.linalg.norm(cropped_ref.affine[:3, :3], axis=0) * (np.array(cropped_ref.shape[:3]) - 1)
    horizontal, vertical = {'x': (1, 2), 'y': (0, 2), 'z': (0, 1)}[selected_display_mode]
    figsize = (fig_height * cropped_extent[horizontal] / cropped_extent[vertical], fig_height)

    for label, input_img, pred in plot_conditions:
        print(label)
        vmin, vmax = np.percentile(input_img.get_fdata(), [low_percentile, high_percentile])

        fig1 = plt.figure(figsize=figsize)
        r = nilearn.plotting.plot_img(crop(input_img, cut_coords, crop_margin),
                                display_mode=selected_display_mode,
                                cut_coords=selected_cut_coords,
                                vmin=vmin,
                                vmax=vmax,
                                **other_plot_opts,
                                figure=fig1)

        fig2 = plt.figure(figsize=figsize)
        s = nilearn.plotting.plot_img(crop(input_img, cut_coords, crop_margin),
                                cut_coords=selected_cut_coords,
                                display_mode=selected_display_mode,
                                vmin=vmin,
                                vmax=vmax,
                                **other_plot_opts,
                                figure=fig2)

        s.add_overlay(crop(pred, cut_coords, crop_margin),
                    interpolation='nearest',
                    resampling_interpolation='nearest',
                    cmap='autumn',
                    alpha=0.5)

        if lesion_mask_uhfadp is not None:
            s.add_contours(crop(lesion_mask_uhfadp, cut_coords, crop_margin),
                        colors='lime',
                        levels=[0.5])

        plt.show()


    surf_plot_margin=crop_margin
    surf_plot_color_white = 'deeppink'
    surf_plot_color_pial = 'magenta'

    # 3T
    q = nilearn.plotting.plot_img(crop(input_3T_coreg, cut_coords, surf_plot_margin),
                                figure=plt.figure(figsize=figsize),
                                display_mode=selected_display_mode,
                                vmin=np.percentile(input_3T_coreg.get_fdata(), low_percentile),
                                vmax=np.percentile(input_3T_coreg.get_fdata(), high_percentile),
                                cut_coords=selected_cut_coords,
                                **other_plot_opts)


    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/lh.pial',
                                    path_lta=affine_3T_path,
                                    color=surf_plot_color_pial)
    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/rh.pial',
                                    path_lta=affine_3T_path,
                                    color=surf_plot_color_pial)
    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/lh.white',
                                    path_lta=affine_3T_path,
                                    color=surf_plot_color_white)
    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/rh.white',
                                    path_lta=affine_3T_path,
                                    color=surf_plot_color_white)


    # UHF default
    r = nilearn.plotting.plot_img(crop(input_uhfdef_coreg, cut_coords, surf_plot_margin),
                                figure=plt.figure(figsize=figsize),
                                display_mode=selected_display_mode,
                                vmin=np.percentile(input_uhfdef_coreg.get_fdata(), low_percentile),
                                vmax=np.percentile(input_uhfdef_coreg.get_fdata(), high_percentile),
                                cut_coords=selected_cut_coords,
                                **other_plot_opts)

    add_surf_overlay_fromfile_coreg(plotting_display=r,
                                    path_surface=f'{meldgraph_dir_uhfdef}output/fs_outputs/{subj_id}/surf/lh.pial',
                                    path_lta=affine_uhfdef_path,
                                    color=surf_plot_color_pial)
    add_surf_overlay_fromfile_coreg(plotting_display=r,
                                    path_surface=f'{meldgraph_dir_uhfdef}output/fs_outputs/{subj_id}/surf/rh.pial',
                                    path_lta=affine_uhfdef_path,
                                    color=surf_plot_color_pial)
    add_surf_overlay_fromfile_coreg(plotting_display=r,
                                    path_surface=f'{meldgraph_dir_uhfdef}output/fs_outputs/{subj_id}/surf/lh.white',
                                    path_lta=affine_uhfdef_path,
                                    color=surf_plot_color_white)
    add_surf_overlay_fromfile_coreg(plotting_display=r,
                                    path_surface=f'{meldgraph_dir_uhfdef}output/fs_outputs/{subj_id}/surf/rh.white',
                                    path_lta=affine_uhfdef_path,
                                    color=surf_plot_color_white)


    # UHF adapted
    p = nilearn.plotting.plot_img(crop(input_uhfadp, cut_coords, surf_plot_margin),
                                figure=plt.figure(figsize=figsize),
                                display_mode=selected_display_mode,
                                vmin=np.percentile(input_uhfadp.get_fdata(), low_percentile),
                                vmax=np.percentile(input_uhfadp.get_fdata(), high_percentile),
                                cut_coords=selected_cut_coords,
                                draw_cross=False,
                                annotate=False,
                                colorbar=False,
                                radiological=True)

    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_uhfadp}output/fs_outputs/{subj_id}/surf/lh.pial',
                                    ras2ras=deoblique_uhfadp,
                                    color=surf_plot_color_pial)
    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_uhfadp}output/fs_outputs/{subj_id}/surf/rh.pial',
                                    ras2ras=deoblique_uhfadp,
                                    color=surf_plot_color_pial)
    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_uhfadp}output/fs_outputs/{subj_id}/surf/lh.white',
                                    ras2ras=deoblique_uhfadp,
                                    color=surf_plot_color_white)
    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_uhfadp}output/fs_outputs/{subj_id}/surf/rh.white',
                                    ras2ras=deoblique_uhfadp,
                                    color=surf_plot_color_white)



    # overlay two surfaces together for visual comparison of registration
    # 3T and UHF default
    p = nilearn.plotting.plot_img(crop(nib.Nifti1Image(np.zeros_like(input_uhfdef_coreg.get_fdata()), input_uhfdef_coreg.affine), cut_coords, surf_plot_margin),
                                figure=plt.figure(figsize=figsize),
                                display_mode=selected_display_mode,
                                vmin=np.percentile(input_uhfdef_coreg.get_fdata(), low_percentile),
                                vmax=np.percentile(input_uhfdef_coreg.get_fdata(), high_percentile),
                                cut_coords=selected_cut_coords,
                                **other_plot_opts)


    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/lh.pial',
                                    path_lta=affine_3T_path,
                                    color=plt.get_cmap('tab10')(0))
    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/rh.pial',
                                    path_lta=affine_3T_path,
                                    color=plt.get_cmap('tab10')(0))
    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/lh.white',
                                    path_lta=affine_3T_path,
                                    color=plt.get_cmap('tab10')(0))
    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/rh.white',
                                    path_lta=affine_3T_path,
                                    color=plt.get_cmap('tab10')(0))

    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_uhfdef}output/fs_outputs/{subj_id}/surf/lh.pial',
                                    path_lta=affine_uhfdef_path,
                                    color=plt.get_cmap('tab10')(1))
    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_uhfdef}output/fs_outputs/{subj_id}/surf/rh.pial',
                                    path_lta=affine_uhfdef_path,
                                    color=plt.get_cmap('tab10')(1))
    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_uhfdef}output/fs_outputs/{subj_id}/surf/lh.white',
                                    path_lta=affine_uhfdef_path,
                                    color=plt.get_cmap('tab10')(1))
    add_surf_overlay_fromfile_coreg(plotting_display=p,
                                    path_surface=f'{meldgraph_dir_uhfdef}output/fs_outputs/{subj_id}/surf/rh.white',
                                    path_lta=affine_uhfdef_path,
                                    color=plt.get_cmap('tab10')(1))


    # 3T and UHF adapted
    q = nilearn.plotting.plot_img(crop(nib.Nifti1Image(np.zeros_like(input_uhfadp.get_fdata()), input_uhfadp.affine), cut_coords, surf_plot_margin),
                                figure=plt.figure(figsize=figsize),
                                display_mode=selected_display_mode,
                                vmin=np.percentile(input_uhfadp.get_fdata(), low_percentile),
                                vmax=np.percentile(input_uhfadp.get_fdata(), high_percentile),
                                cut_coords=selected_cut_coords,
                                **other_plot_opts)

    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/lh.pial',
                                    path_lta=affine_3T_path,
                                    color=plt.get_cmap('tab10')(0))
    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/rh.pial',
                                    path_lta=affine_3T_path,
                                    color=plt.get_cmap('tab10')(0))
    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/lh.white',
                                    path_lta=affine_3T_path,
                                    color=plt.get_cmap('tab10')(0))
    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_3T}output/fs_outputs/{subj_id}/surf/rh.white',
                                    path_lta=affine_3T_path,
                                    color=plt.get_cmap('tab10')(0))

    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_uhfadp}output/fs_outputs/{subj_id}/surf/lh.pial',
                                    ras2ras=deoblique_uhfadp,
                                    color=plt.get_cmap('tab10')(1))
    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_uhfadp}output/fs_outputs/{subj_id}/surf/rh.pial',
                                    ras2ras=deoblique_uhfadp,
                                    color=plt.get_cmap('tab10')(1))
    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_uhfadp}output/fs_outputs/{subj_id}/surf/lh.white',
                                    ras2ras=deoblique_uhfadp,
                                    color=plt.get_cmap('tab10')(1))
    add_surf_overlay_fromfile_coreg(plotting_display=q,
                                    path_surface=f'{meldgraph_dir_uhfadp}output/fs_outputs/{subj_id}/surf/rh.white',
                                    ras2ras=deoblique_uhfadp,
                                    color=plt.get_cmap('tab10')(1))